# Iterative debiasing at every layer

For **each** of the 24 layers independently: applies N fresh CAV cycles in sequence.
At each cycle a **new** CAV is trained on the already-projected activations, then projected out.

```
For each layer L in 0..23:
  X₀ = raw activations at layer L
  for i in 1..N:
    train new CAVᵢ on Xᵢ₋₁  →  project  →  Xᵢ
  propagate X_N through layers L+1..23  →  model output
```

Compare with:
- **`03_iterative_single_layer`**: same logic but for one fixed layer only
- **`04_fixed_multi_layer`**: one CAV trained once, applied at k consecutive layers

Here each layer gets its own N fresh CAVs — so the model cannot recover the concept
by "hiding" it in a subspace that a single CAV misses.

**Methods:** `lr` and `pclarc` only.  
diff_means direction equals the normalised class-mean difference; after one orthogonal
projection that direction becomes zero, making re-training undefined.

Output structure:
```
data/activations/debiased/iterative_all_layers/{CONCEPT}/{method}/layer_{L:02d}/
  train/model_output.parquet
  test/ model_output.parquet
  cavs.csv     # train_acc + test_acc per iteration at this layer
  info.json
```

In [ ]:
import json
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image
from dotenv import load_dotenv
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
load_dotenv()

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from software.dataset import get_concept_split, DS_SIZE
from software.torch_lr import TorchLR

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────────────
CONCEPT            = 'eyeglasses'
N_DEBIAS_PER_LAYER = 2          # fresh CAV cycles applied at each layer
METHODS            = ['lr', 'pclarc']   # diff_means excluded: zero direction after 1st projection
GPU_BATCH_SIZE     = 64
NUM_WORKERS        = 8
MODEL_ID           = 'openai/clip-vit-large-patch14'
PARQUET_COMPRESSION = 'snappy'
NUM_LAYERS         = 24

SOLVER   = 'torch_lr'
SOLVER_C = 0.1

METADATA_PATH = ROOT / 'data' / 'metadata.csv'
IMAGES_DIR    = ROOT / 'data' / 'images'
RAW_DIR       = ROOT / 'data' / 'activations' / 'raw'
DATA_OUT      = ROOT / 'data' / 'activations' / 'debiased' / 'iterative_all_layers' / CONCEPT
PLOT_DIR      = ROOT / 'notebooks' / 'results' / 'multiple_debias' / CONCEPT / 'iterative_all_layers'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

assert METADATA_PATH.exists(), f'Missing: {METADATA_PATH}'
assert RAW_DIR.exists(),       f'Run 01_get_activations.ipynb first'

df_train = get_concept_split(CONCEPT, 'train', metadata_path=METADATA_PATH)[['filename', CONCEPT]]
df_test  = get_concept_split(CONCEPT, 'test',  metadata_path=METADATA_PATH)[['filename', CONCEPT]]

print(f'Concept            : {CONCEPT}')
print(f'N debias per layer : {N_DEBIAS_PER_LAYER}')
print(f'Methods            : {METHODS}')
print(f'Train              : {len(df_train)} images  (target: {DS_SIZE * 7})')
print(f'Test               : {len(df_test)} images  (target: {DS_SIZE})')
print(f'Output             : {DATA_OUT}')

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype    = torch.bfloat16 if device == 'cuda' else torch.float32

processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, token=HF_TOKEN,
).to(device).eval()
print(f'Model: {MODEL_ID} | {device} | {dtype}')


class CelebADataset(Dataset):
    def __init__(self, df, images_dir, processor):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.processor  = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.images_dir / row['filename']).convert('RGB')
        px  = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        return px, row['filename'], int(row[CONCEPT])


def make_loader(df):
    return DataLoader(
        CelebADataset(df, IMAGES_DIR, processor),
        batch_size=GPU_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device == 'cuda'),
        persistent_workers=(NUM_WORKERS > 0),
    )


loader_train = make_loader(df_train)
loader_test  = make_loader(df_test)

In [ ]:
def make_cav_clf():
    if SOLVER == 'torch_lr':
        return TorchLR(C=SOLVER_C, max_iter=500, random_state=42)
    return SGDClassifier(
        loss='log_loss', penalty='l2', alpha=1.0 / (2.0 * SOLVER_C),
        max_iter=1000, tol=1e-4, random_state=42,
    )


def load_raw_layer(split_label, layer_idx):
    """Load raw activations at layer_idx for the concept-balanced subset."""
    split_df = df_train if split_label == 'train' else df_test
    df_raw = pd.read_parquet(RAW_DIR / split_label / f'layer_{layer_idx:02d}.parquet')
    df_raw = df_raw.merge(split_df, on='filename', how='inner')
    feat_cols = [c for c in df_raw.columns if c not in ('filename', CONCEPT)]
    return (
        df_raw[feat_cols].values.astype(np.float32),
        df_raw['filename'].tolist(),
        df_raw[CONCEPT].values.astype(int),
    )


def train_cav(X, y, method):
    """Train CAV on current activations. Returns (cav, meta) or (None, None) if degenerate.

    meta keys: acc, threshold, target_val, lr_intercept, clf
    """
    if method == 'lr':
        clf = make_cav_clf()
        clf.fit(X, y)
        w    = clf.coef_[0].astype(np.float64)
        norm = np.linalg.norm(w)
        if norm < 1e-10:
            return None, None
        cav  = (w / norm).astype(np.float32)
        b    = float(clf.intercept_[0]) if hasattr(clf, 'intercept_') else 0.0
        lr_intercept = b / norm
        return cav, {
            'acc':          clf.score(X, y),
            'threshold':    -lr_intercept,
            'target_val':   None,
            'lr_intercept': lr_intercept,
            'clf':          clf,
        }

    # pclarc: diff_means direction + translation to neg-class mean
    X64  = X.astype(np.float64)
    diff = X64[y == 1].mean(0) - X64[y == 0].mean(0)
    norm = np.linalg.norm(diff)
    if norm < 1e-10:
        return None, None
    cav  = (diff / norm).astype(np.float32)
    proj = X64 @ cav.astype(np.float64)
    thr  = float((proj[y == 1].mean() + proj[y == 0].mean()) / 2)
    tv   = float(proj[y == 0].mean())
    acc  = accuracy_score(y, (proj > thr).astype(int))
    return cav, {
        'acc':          acc,
        'threshold':    thr,
        'target_val':   tv,
        'lr_intercept': None,
        'clf':          None,
    }


def project(X, cav, method, target_val=None):
    X64 = X.astype(np.float64)
    c64 = cav.astype(np.float64)
    if method == 'pclarc':
        return (X64 + np.outer(target_val - X64 @ c64, c64)).astype(np.float32)
    return (X64 - np.outer(X64 @ c64, c64)).astype(np.float32)


def save_parquet(X, filenames, path):
    df = pd.DataFrame(X.astype(np.float16))
    df.columns = df.columns.astype(str)
    df.insert(0, 'filename', filenames)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, compression=PARQUET_COMPRESSION, index=False)


def run_propagate(loader, X_injected, filenames_order, debias_layer):
    """Inject X_injected at debias_layer CLS output, propagate through the remaining
    layers, capture and return the final 768-dim model output.
    """
    X_t   = torch.from_numpy(X_injected).to(device=device, dtype=dtype)
    enc   = model.vision_model.encoder.layers
    out_buf, fn_all = [], []
    ptr   = [0]

    def inject(module, input, output):
        is_tuple = isinstance(output, tuple)
        hidden   = output[0] if is_tuple else output
        bs       = hidden.shape[0]
        rows     = X_t[ptr[0]:ptr[0] + bs]
        h_new    = hidden.clone()
        h_new[:, 0, :] = rows.to(hidden.dtype)
        ptr[0]  += bs
        return (h_new,) + output[1:] if is_tuple else h_new

    def capture(module, input, output):
        out_buf.append(output.detach().float().cpu().numpy())

    h1 = enc[debias_layer].register_forward_hook(inject)
    h2 = model.visual_projection.register_forward_hook(capture)
    try:
        with torch.no_grad():
            for pixels, fnames, _ in loader:
                model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
                fn_all.extend(fnames)
    finally:
        h1.remove()
        h2.remove()

    return np.concatenate(out_buf, axis=0), fn_all

In [ ]:
for method in METHODS:
    print(f'\n=== Method: {method} ===')

    for layer in tqdm(range(NUM_LAYERS), desc=method):
        out_base = DATA_OUT / method / f'layer_{layer:02d}'
        if (out_base / 'info.json').exists():
            continue

        X_tr, fn_tr, y_tr = load_raw_layer('train', layer)
        X_te, fn_te, y_te = load_raw_layer('test',  layer)

        X_cur_tr = X_tr.copy()
        X_cur_te = X_te.copy()
        cav_records  = []
        final_cav    = None
        final_meta   = None

        for it in range(1, N_DEBIAS_PER_LAYER + 1):
            cav, meta = train_cav(X_cur_tr, y_tr, method)
            if cav is None:
                break

            # measure test accuracy BEFORE projecting (honest pre-projection estimate)
            if method == 'lr':
                acc_te = float(meta['clf'].score(X_cur_te, y_te))
            else:
                proj_te = X_cur_te.astype(np.float64) @ cav.astype(np.float64)
                acc_te  = accuracy_score(y_te, (proj_te > meta['threshold']).astype(int))

            X_cur_tr = project(X_cur_tr, cav, method, meta['target_val'])
            X_cur_te = project(X_cur_te, cav, method, meta['target_val'])

            cav_records.append({
                'layer':      layer,
                'iteration':  it,
                'train_acc':  meta['acc'],
                'test_acc':   acc_te,
                'threshold':  meta['threshold'],
                'target_val': meta['target_val'],
            })
            final_cav, final_meta = cav, meta

        if final_cav is None:
            continue

        out_tr, fn_tr2 = run_propagate(loader_train, X_cur_tr, fn_tr, layer)
        out_te, fn_te2 = run_propagate(loader_test,  X_cur_te, fn_te, layer)

        save_parquet(out_tr, fn_tr2, out_base / 'train' / 'model_output.parquet')
        save_parquet(out_te, fn_te2, out_base / 'test'  / 'model_output.parquet')

        pd.DataFrame(cav_records).to_csv(out_base / 'cavs.csv', index=False)

        info = {
            'concept':            CONCEPT,
            'method':             method,
            'debias_type':        'iterative_all_layers',
            'debias_layer':       layer,
            'n_debias_per_layer': N_DEBIAS_PER_LAYER,
            'actual_iterations':  len(cav_records),
            'model_id':           MODEL_ID,
            'created_at':         datetime.now(timezone.utc).isoformat(),
            'description': (
                f"Iterative debiasing of '{CONCEPT}' at layer {layer} using {method}. "
                f"{len(cav_records)} CAV cycle(s) applied; each cycle trains a fresh CAV "
                f"on the current projected activations and projects again. "
                f"model_output = 768-dim CLIP embedding after propagating final activations."
            ),
        }
        with open(out_base / 'info.json', 'w') as f:
            json.dump(info, f, indent=2)

print('\nDone.')

## Visualization: CAV accuracy per layer and iteration

In [ ]:
# Heatmap: CAV test accuracy per (layer x iteration) for each method.
# Shows where the concept is detectable and whether each additional iteration
# still finds residual signal.

for method in METHODS:
    records = []
    for layer in range(NUM_LAYERS):
        p = DATA_OUT / method / f'layer_{layer:02d}' / 'cavs.csv'
        if not p.exists():
            continue
        df = pd.read_csv(p)
        for _, row in df.iterrows():
            records.append({
                'layer':     int(row['layer']),
                'iteration': int(row['iteration']),
                'test_acc':  float(row['test_acc']),
                'train_acc': float(row['train_acc']),
            })

    if not records:
        print(f'[{method}] No data found — run pipeline first.')
        continue

    df_all = pd.DataFrame(records)
    pivot  = df_all.pivot(index='layer', columns='iteration', values='test_acc')

    fig, ax = plt.subplots(figsize=(max(4, N_DEBIAS_PER_LAYER * 1.5 + 2), 7))
    sns.heatmap(
        pivot, ax=ax, cmap='RdYlGn_r', vmin=0.5, vmax=1.0,
        annot=True, fmt='.2f', linewidths=0.4, cbar_kws={'label': 'CAV test accuracy'},
    )
    ax.set_xlabel('Iteration (CAV cycle within this layer)')
    ax.set_ylabel('Layer')
    ax.set_title(
        f'CAV test accuracy before each projection — method: {method}, concept: "{CONCEPT}"\n'
        f'(green = near-random / concept removed,  red = concept still detectable)',
        fontsize=10,
    )
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'cav_acc_per_layer_iter_{method}.png', dpi=150)
    plt.show()

In [ ]:
# Line plot: accuracy at iteration 1 vs iteration N per layer, per method.
# Shows how much additional gain each extra CAV cycle provides at each layer.

fig, axes = plt.subplots(1, len(METHODS), figsize=(6 * len(METHODS), 4), sharey=True)
if len(METHODS) == 1:
    axes = [axes]

for ax, method in zip(axes, METHODS):
    records = []
    for layer in range(NUM_LAYERS):
        p = DATA_OUT / method / f'layer_{layer:02d}' / 'cavs.csv'
        if not p.exists():
            continue
        df = pd.read_csv(p)
        for it, grp in df.groupby('iteration'):
            records.append({'layer': int(grp['layer'].iloc[0]),
                            'iteration': int(it),
                            'test_acc': float(grp['test_acc'].iloc[0])})

    if not records:
        ax.set_title(f'{method}: no data')
        continue

    df_m = pd.DataFrame(records)
    for it, grp in df_m.groupby('iteration'):
        grp_s = grp.sort_values('layer')
        ax.plot(grp_s['layer'], grp_s['test_acc'], marker='o', ms=3,
                label=f'iter {int(it)}')

    ax.axhline(0.5, color='red', lw=0.8, ls=':', label='chance')
    ax.set_xlabel('Layer')
    ax.set_ylabel('CAV test accuracy (before projection)')
    ax.set_title(f'Method: {method}', fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)

fig.suptitle(
    f'CAV test accuracy per layer and iteration — concept: "{CONCEPT}"\n'
    f'(each line = one CAV cycle; lower = concept already removed by previous cycle)',
    fontsize=11,
)
plt.tight_layout()
plt.savefig(PLOT_DIR / f'cav_acc_per_layer_iter_lines.png', dpi=150)
plt.show()